# S5 Attention Weight Analysis — SoftmaxAttentionFusion vs EMG SNR

**실행 순서**
1. 셀 1: Drive 마운트 & 패키지 설치
2. 셀 2: 어텐션 가중치 추출 (52개 체크포인트 → ~10분)
3. 셀 3: Spearman 상관 분석
4. 셀 4: 시각화 (scatter plot + bar chart)

In [1]:
# ── 셀 1: Drive 마운트 & 설치 ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'h5py', 'scipy', 'scikit-learn'])

DRIVE_ROOT = '/content/drive/MyDrive/BCI_Research'
shutil.copy(f'{DRIVE_ROOT}/src/attention_analysis.py',
            '/content/attention_analysis.py')

import subprocess
gpu = subprocess.getoutput('nvidia-smi --query-gpu=name --format=csv,noheader')
print(f'Drive 마운트 완료 | GPU: {gpu}')

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/BCI_Research/src/attention_analysis.py'

In [ ]:
# ── 셀 2: 어텐션 가중치 추출 ────────────────────────────────────
# 52개 체크포인트에서 w_eeg / w_emg 추출 (예상 ~10분)
!python /content/attention_analysis.py \
    --drive_root /content/drive/MyDrive/BCI_Research

In [ ]:
# ── 셀 3: 결과 로드 & 상관 요약 출력 ───────────────────────────
import pandas as pd, json

OUT = f'{DRIVE_ROOT}/results/attention'

df = pd.read_csv(f'{OUT}/attention_weights_per_subject.csv')
print('=== 피험자별 어텐션 가중치 (상위 10명) ===')
print(df[['sid','w_eeg_mean','w_emg_mean','emg_snr_db','accuracy']]
      .sort_values('w_emg_mean', ascending=False)
      .head(10)
      .to_string(index=False))

corr = json.load(open(f'{OUT}/attention_snr_correlation.json'))
print('\n=== Spearman 상관 결과 ===')
for key, v in corr.items():
    print(f"  {v['label']:<45}: ρ={v['rho']:+.4f}  p={v['p']:.4f}  {v['sig']}")

In [ ]:
# ── 셀 4: 시각화 ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) w_emg vs EMG SNR
ax = axes[0]
ax.scatter(df['emg_snr_db'], df['w_emg_mean'],
           c='#DD8452', alpha=0.7, edgecolors='k', linewidths=0.4)
rho, p = spearmanr(df['emg_snr_db'].dropna(), df['w_emg_mean'].loc[df['emg_snr_db'].dropna().index])
ax.set_xlabel('EMG SNR (dB)', fontsize=11)
ax.set_ylabel('Mean w_emg', fontsize=11)
ax.set_title(f'w_emg vs EMG SNR\n(ρ={rho:.3f}, p={p:.3f})', fontsize=11)
ax.grid(alpha=0.3)
# 추세선
z = np.polyfit(df['emg_snr_db'].dropna(),
               df['w_emg_mean'].loc[df['emg_snr_db'].dropna().index], 1)
xfit = np.linspace(df['emg_snr_db'].min(), df['emg_snr_db'].max(), 100)
ax.plot(xfit, np.polyval(z, xfit), 'r--', linewidth=1.5)

# (b) w_emg vs Accuracy
ax = axes[1]
ax.scatter(df['accuracy'], df['w_emg_mean'],
           c='#55A868', alpha=0.7, edgecolors='k', linewidths=0.4)
rho2, p2 = spearmanr(df['accuracy'], df['w_emg_mean'])
ax.set_xlabel('Classification Accuracy', fontsize=11)
ax.set_ylabel('Mean w_emg', fontsize=11)
ax.set_title(f'w_emg vs Accuracy\n(ρ={rho2:.3f}, p={p2:.3f})', fontsize=11)
ax.grid(alpha=0.3)
z2 = np.polyfit(df['accuracy'], df['w_emg_mean'], 1)
xfit2 = np.linspace(df['accuracy'].min(), df['accuracy'].max(), 100)
ax.plot(xfit2, np.polyval(z2, xfit2), 'r--', linewidth=1.5)

# (c) w_eeg / w_emg distribution across subjects
ax = axes[2]
sids = df['sid'].values
ax.bar(sids - 0.2, df['w_eeg_mean'], width=0.4,
       label='w_eeg', color='#4C72B0', alpha=0.7)
ax.bar(sids + 0.2, df['w_emg_mean'], width=0.4,
       label='w_emg', color='#DD8452', alpha=0.7)
ax.axhline(0.5, color='k', linestyle='--', linewidth=0.8, label='equal weight')
ax.set_xlabel('Subject ID', fontsize=11)
ax.set_ylabel('Attention Weight', fontsize=11)
ax.set_title('EEG vs sEMG Attention per Subject', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

fig.suptitle('SoftmaxAttentionFusion Weight Analysis (52 subjects)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT}/attention_analysis_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: attention_analysis_plot.png')